# Orbital Debris SQL Queries

This notebook is intentionally SQL-first and contains query work only (no visualizations).

## **Setup And Function Declarations**

In [ ]:
import pandas as pd
import sqlite3

def run_query(sql):
    return pd.read_sql(sql, conn)

def query_all_satellites():
    query = """
    SELECT
      norad_id,
      cospar_id,
      object_name,
      launch_year,
      launch_date,
      launch_site,
      owner,
      country_operator,
      object_type
    FROM satellites
    JOIN launch_events ON launch_events.launch_id = satellites.launch_id
    JOIN ownership_operators ON ownership_operators.owner_code = satellites.owner_code;
    """
    return run_query(query)

# Gets a single satellite's information based on NORAD ID
def query_satellite_info(norad_id):
    query = f"""
    SELECT
      norad_id,
      cospar_id,
      object_name,
      launch_year,
      launch_date,
      launch_site,
      owner,
      country_operator,
      object_type
    FROM satellites
    JOIN launch_events ON launch_events.launch_id = satellites.launch_id
    JOIN ownership_operators ON ownership_operators.owner_code = satellites.owner_code
    WHERE norad_id = {norad_id};
    """
    return run_query(query)

def query_satellite_info_list(norad_ids):
    # This will create a comma-separated string of NORAD IDs for the SQL IN clause
    ids = ','.join(str(id) for id in norad_ids)
    
    query = f"""
    SELECT
      norad_id,
      cospar_id,
      object_name,
      launch_year,
      launch_date,
      launch_site,
      owner,
      country_operator,
      object_type
    FROM satellites
    JOIN launch_events ON launch_events.launch_id = satellites.launch_id
    JOIN ownership_operators ON ownership_operators.owner_code = satellites.owner_code
    WHERE norad_id IN ({ids});
    """
    print(f"Querying information for NORAD IDs: {norad_ids}")
    print(f"Executing SQL query:\n{query}")
    return run_query(query)

pd.set_option('display.max_rows', 100)

conn = sqlite3.connect('../data/clean/orbital_debris.db')

In [ ]:
df = query_satellite_info(25544)
display(df)

df = query_satellite_info_list([25544, 33591, 43013])
display(df)

df = query_all_satellites()
display(df)

## Primary Question 1: Growth and Decoupling Baseline

In [ ]:
# I absolutely detest table aliases in SQL. I find them to be more confusing 
# than helpful, especially when the table names are not that long to begin with.
# I understand that they can be useful in some cases, but I prefer to just write 
# out the full table names for clarity. Nested queries are usually the only time
# I find them to be necessary, and even then I try to avoid them if possible.

q1 = '''
WITH yearly AS (
  SELECT
    launch_events.launch_year AS launch_year,
    COUNT(*) AS objects_launched,
    COUNT(DISTINCT launch_events.launch_id) AS launch_missions,
    SUM(
      CASE
        WHEN UPPER(COALESCE(satellites.object_type, '')) = 'PAYLOAD' THEN 1
        ELSE 0
      END
    ) AS payload_objects
  FROM satellites
  JOIN launch_events ON launch_events.launch_id = satellites.launch_id
  WHERE launch_events.launch_year IS NOT NULL
  GROUP BY launch_events.launch_year
)
SELECT
  launch_year,
  objects_launched,
  launch_missions,
  payload_objects,
  ROUND(100.0 * payload_objects / NULLIF(objects_launched, 0), 2) AS payload_share_pct,
  SUM(objects_launched) OVER (ORDER BY launch_year) AS cumulative_objects
FROM yearly
ORDER BY launch_year;
'''
launch_trend = run_query(q1)
launch_trend.to_csv('../data/clean/queries/pq1_launch_trend.csv', index=False)
launch_trend.head(100)

## **Primary Question 2: High-Risk Distribution by Altitude Band**

## **Primary Question 3: Zombie Concentration by Owner**

## **Secondary: Object Type × Operational Status**

## **Secondary: User Category Profile**

## **Extra Questions!**

## **Tidy Up!**

In [ ]:
# Close the connection.
# Release resources and ensure clean exit (old habits die hard).
conn.close()
print('Connection closed.')